# kelv_helm1: Kelvin-Helmholtz instability of a stratified shear layer
pyOM2: for_config/kelv_helm1.f90, py_config/kelv_helm1.py, for_config/shear1.f90

Non-hydrostatic x-z slice (nh_simple). Velocity and buoyancy change across a thin
interface in the middle of the domain. With sponge=True the profiles are restored
in the first eighth of the periodic domain (kelv_helm1), with sponge=False the
shear layer evolves freely (shear1).

In [ ]:
import os, sys
sys.path.insert(0, os.environ.get('PYOM3_ROOT', os.path.abspath('../..')))
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import pyOM3 as OM
from pyOM3.nh_simple.model import model

In [ ]:
fac        = 2.0
nx, ny, nz = int(1.5*64*fac), 1, int(40*fac)
dx = dz    = 0.25/fac
dt         = 0.04/fac
mix        = 5e-3/fac**2
z_mid, z_bot = -nz//2*dz, -(nz - 0.5)*dz
grav, beta_T = 9.81, 1.67e-4        # buoyancy b = grav*beta_T*(T - 10)
sponge     = True
t_rest     = 1./(15*dt)
nsteps     = 1500                   # 60 s
plot_every = 100

par = dict(nx=nx, ny=ny, nz=nz, dx=dx, dy=dx, dz=dz, dt=dt,
           f0=0., N0=0., Ro=1., dsqr=1., periodic_in_z=False,
           Ah=mix, Av=mix, buoyancy_superbee_advection=True)

In [ ]:
def T_star(z):
    return 9.85 - 6.5*np.tanh((z - z_mid)/z_bot*100)

def u_star(z):
    return 0.6 + 0.5*np.tanh((z - z_mid)/z_bot*100)


class kelv_helm1(model):

    def set_forcing(self, tau):
        if not sponge: return
        self.du = OM.add_to_array(self.du, (tau,), self.rate*(self.u_star - self.u))
        self.db = OM.add_to_array(self.db, (tau,), self.rate*(self.b_star - self.b))

In [ ]:
M = kelv_helm1(par)
M.setup()
# local coordinates with halo points, arrays are (x,y,z), z = 0 at the surface
h = M.halo
x = (np.arange(M.Nx) - h + M.xs_pe - 0.5)*dx
y = (np.arange(M.Ny) - h + M.ys_pe - 0.5)*dx
z = (np.arange(M.Nz) - h + M.zs_pe - 0.5)*dz - nz*dz
X, Y, Z = np.meshgrid(x, y, z, indexing='ij')
M.u_star = OM.np.asarray(u_star(Z))
M.b_star = OM.np.asarray(grav*beta_T*(T_star(Z) - 10.))
M.rate   = OM.np.asarray(np.where(X < nx/8*dx, t_rest, 0.))
M.u = M.apply_bc(M.u_star*1)
M.b = M.apply_bc(M.b_star + grav*beta_T*1e-3*z_bot*np.sin(X/(20*dx)*np.pi))

In [ ]:
def plot(t):
    clear_output(wait=True)
    T = M.b[h:-h, h, h:-h].T/(grav*beta_T) + 10.
    plt.figure(figsize=(9, 4))
    plt.pcolormesh(x[h:-h], z[h:-h], T, cmap='RdBu_r')
    plt.colorbar(label='T [deg C]'); plt.xlabel('x [m]'); plt.ylabel('z [m]')
    plt.title('t = %.1f s' % t); plt.show()

In [ ]:
A, B, C = 1., 0., 0.                 # Adams-Bashforth weights, forward Euler at the start
tau, taum1, taum2 = 2, 1, 0          # pointers to the time levels
for n in range(nsteps):
    M.time_step(A, B, C, tau, taum1, taum2)
    if n % plot_every == 0: plot(n*dt)
    tau, taum1, taum2 = (tau+1) % 3, (taum1+1) % 3, (taum2+1) % 3
    A, B, C = (1.5, -0.5, 0.) if n == 0 else (23./12, -16./12, 5./12)

In [ ]:
plot(nsteps*dt)